<a href="https://colab.research.google.com/github/fernandanasc/projeto-bd-estacionamento/blob/main/estacionamento_hospitalar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema de Estacionamento Hospitalar – Projeto de BD (Iteração 2)

**IMD0401 – Banco de Dados – T02 (2026.2)** · Docente: Gustavo Alves Bezerra
Discente: Fernanda Vitoria Silva Do Nascimento

Este notebook **cria**, **popula** e **usa** o banco de dados do estacionamento de um hospital.

**Minimundo (resumo)**
- **Avulso (uso diário):** pega um cartão na entrada, paga o tempo de permanência na saída e devolve o cartão na máquina. Não é cadastrado como pessoa; fica registrado o veículo e a estadia.
- **Mensalista (cliente):** cadastrado com nome, CPF, telefones e endereço. Tem cartão pessoal (a máquina devolve o cartão na saída) e paga **mensalidade**.
- **Categoria isenta:** ambulâncias e veículos médicos não pagam.
- Toda entrada e saída é registrada em uma **estadia**, com a guarita usada.

**Roteiro**
1. Instalação do PostgreSQL
2. Criação do banco (tipos, tabelas e restrições)
3. População com dados de exemplo
4. Testes das restrições
5. Consultas (exemplos de utilização)

O DER está em `diagramas/der_v2.png` e o modelo lógico em `diagramas/dbdiagram_v2.dbml`.

## 1. Instalação do PostgreSQL

Mesmo procedimento das aulas: instalar o PostgreSQL na máquina do Colab.

In [1]:
# instalação do postgresql na máquina do colab (saída escondida para não poluir o notebook)
!sudo apt update -qq > /dev/null 2>&1
!sudo apt install -y -qq postgresql postgresql-contrib > /dev/null 2>&1

Verificar se foi instalado corretamente e iniciar o servidor.

In [2]:
!psql --version
!sudo service postgresql start
!sudo service postgresql status

psql (PostgreSQL) 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1)
[ OK ]
16/main (port 5432): online


Criar o banco de dados `estacionamento`.

In [3]:
!sudo -u postgres createdb estacionamento

O Colab usa o horário UTC. Ajustamos o banco para o fuso de Natal, para que `NOW()` mostre a hora local.

In [4]:
!sudo -u postgres psql -c "ALTER DATABASE estacionamento SET TIMEZONE TO 'America/Fortaleza'"

ALTER DATABASE


Criar o comando `posql` já apontando para o banco `estacionamento`.

- `-d estacionamento`: sempre conecta no nosso banco
- `-P pager=off`: desliga o paginador, então não precisa digitar `\q`

In [5]:
!echo '#!/bin/bash' > /usr/local/bin/posql
!echo 'sudo -u postgres psql -d estacionamento -P pager=off "$@"' >> /usr/local/bin/posql
!chmod +x /usr/local/bin/posql
!posql -c "SELECT CURRENT_DATABASE();"

 current_database 
------------------
 estacionamento
(1 row)



## 2. Criação do banco

### Do DER para as tabelas

| No DER | No PostgreSQL |
|---|---|
| Atributo chave | `PRIMARY KEY` |
| Atributo composto (`endereco`) | `CREATE TYPE ... AS (...)` |
| Atributo multivalorado (`telefone`) | array `TEXT[]` |
| Atributo derivado (`status`, `tempo_permanencia`) | não é guardado; é calculado na consulta (`VIEW`) |
| Domínio restrito (`tipo`, `status`, `forma_pagamento`) | `CREATE TYPE ... AS ENUM` |
| Relacionamento 1:N | chave estrangeira (`REFERENCES`) do lado N |
| Relacionamento 1:1 | chave estrangeira com `UNIQUE` |
| Participação total | chave estrangeira `NOT NULL` |
| Participação parcial | chave estrangeira que aceita `NULL` |

### 2.1 Tipos

**Atributo composto:** o endereço é formado por logradouro, número, bairro, cidade, UF e CEP. Como visto em aula, isso vira um tipo novo (parecido com um `struct` do C).

In [6]:
%%bash
posql -c "
CREATE TYPE endereco AS (
  logradouro TEXT,
  numero INTEGER,
  bairro TEXT,
  cidade TEXT,
  uf CHAR(2),
  cep CHAR(8)
);
"

CREATE TYPE


**Domínios restritos:** o tipo e o status do cartão e a forma de pagamento só podem assumir alguns valores. Usamos `ENUM`, como no exemplo do tipo sanguíneo visto em aula.

In [7]:
%%bash
posql -c "
CREATE TYPE tipo_cartao AS ENUM ('MENSALISTA', 'AVULSO');
CREATE TYPE status_cartao AS ENUM ('ATIVO', 'BLOQUEADO', 'PERDIDO');
CREATE TYPE forma_pagamento AS ENUM ('DINHEIRO', 'CARTAO', 'PIX');
"

CREATE TYPE
CREATE TYPE
CREATE TYPE


In [8]:
!posql -c "\dT"

           List of data types
 Schema |      Name       | Description 
--------+-----------------+-------------
 public | endereco        | 
 public | forma_pagamento | 
 public | status_cartao   | 
 public | tipo_cartao     | 
(4 rows)



### 2.2 Tabelas de classificação dos veículos

Criadas primeiro porque as outras tabelas fazem referência a elas.

- **FABRICANTE (1) – FABRICA – (N) MODELO:** todo modelo tem fabricante (participação total), então `cod_fabricante` é `NOT NULL`.
- **CATEGORIA:** o atributo `isenta` indica os veículos hospitalares e médicos, que não pagam.

In [9]:
%%bash
posql -c "
CREATE TABLE fabricante (
  cod_fabricante SERIAL PRIMARY KEY,
  nome_fabricante TEXT NOT NULL UNIQUE
);

CREATE TABLE modelo (
  cod_modelo SERIAL PRIMARY KEY,
  nome_modelo TEXT NOT NULL,
  cod_fabricante INTEGER NOT NULL REFERENCES fabricante(cod_fabricante)
);

CREATE TABLE categoria (
  cod_categoria SERIAL PRIMARY KEY,
  descricao TEXT NOT NULL UNIQUE,
  isenta BOOLEAN NOT NULL DEFAULT FALSE
);
"

CREATE TABLE
CREATE TABLE
CREATE TABLE


### 2.3 Cliente (mensalista)

- `telefone TEXT[]`: multivalorado, um cliente pode ter vários telefones.
- `endereco endereco`: composto, usa o tipo criado acima.
- `CHECK` no CPF: exatamente 11 dígitos.

In [10]:
%%bash
posql -c "
CREATE TABLE cliente (
  cod_cliente SERIAL PRIMARY KEY,
  nome TEXT NOT NULL,
  cpf CHAR(11) NOT NULL UNIQUE CHECK (cpf ~ '^[0-9]{11}$'),
  telefone TEXT[],
  endereco endereco
);
"

CREATE TABLE


### 2.4 Veículo

- A **placa** é a chave natural. O `CHECK` aceita o padrão antigo (`ABC1234`) e o Mercosul (`ABC1D23`).
- **MODELO – INSTANCIA – VEICULO** e **CATEGORIA – CLASSIFICA – VEICULO:** participação total, então `NOT NULL`.
- **CLIENTE – POSSUI – VEICULO:** participação parcial dos dois lados (o veículo avulso não tem dono cadastrado), então `cod_cliente` aceita `NULL`.

In [11]:
%%bash
posql -c "
CREATE TABLE veiculo (
  placa CHAR(7) PRIMARY KEY CHECK (placa ~ '^[A-Z]{3}[0-9][A-Z0-9][0-9]{2}$'),
  cor TEXT,
  ano SMALLINT CHECK (ano BETWEEN 1950 AND 2030),
  cod_modelo INTEGER NOT NULL REFERENCES modelo(cod_modelo),
  cod_categoria INTEGER NOT NULL REFERENCES categoria(cod_categoria),
  cod_cliente INTEGER REFERENCES cliente(cod_cliente)
);
"

CREATE TABLE


### 2.5 Cartão de acesso

**CLIENTE – RECEBE – CARTAO_ACESSO** é parcial dos dois lados, porque o cartão avulso não tem dono. Para garantir a regra do minimundo, usamos um `CHECK`:

- cartão **MENSALISTA** precisa ter cliente e data de validade;
- cartão **AVULSO** não pode ter cliente nem validade.

In [12]:
%%bash
posql -c "
CREATE TABLE cartao_acesso (
  num_cartao INTEGER PRIMARY KEY,
  tipo tipo_cartao NOT NULL,
  status status_cartao NOT NULL DEFAULT 'ATIVO',
  data_emissao DATE NOT NULL,
  data_validade DATE,
  cod_cliente INTEGER REFERENCES cliente(cod_cliente),
  CONSTRAINT cartao_tipo_cliente CHECK (
    (tipo = 'MENSALISTA' AND cod_cliente IS NOT NULL AND data_validade IS NOT NULL)
    OR
    (tipo = 'AVULSO' AND cod_cliente IS NULL AND data_validade IS NULL)
  ),
  CONSTRAINT validade_depois_emissao CHECK (data_validade > data_emissao)
);
"

CREATE TABLE


### 2.6 Guarita

In [13]:
%%bash
posql -c "
CREATE TABLE guarita (
  cod_guarita SERIAL PRIMARY KEY,
  localizacao TEXT NOT NULL
);
"

CREATE TABLE


### 2.7 Estadia

Cada entrada no pátio gera uma estadia.

- **UTILIZA** (cartão), **REALIZA** (veículo) e **REGISTRA_ENTRADA** (guarita): participação total, então `NOT NULL`.
- **REGISTRA_SAIDA:** parcial, porque enquanto o veículo está no pátio a saída fica vazia.
- Entre **GUARITA** e **ESTADIA** há **dois** relacionamentos, por isso existem duas chaves estrangeiras para a mesma tabela.
- `CHECK`: a saída tem que ser depois da entrada, e a data de saída e a guarita de saída são preenchidas juntas.

In [14]:
%%bash
posql -c "
CREATE TABLE estadia (
  cod_estadia SERIAL PRIMARY KEY,
  data_hora_entrada TIMESTAMP NOT NULL,
  data_hora_saida TIMESTAMP,
  num_cartao INTEGER NOT NULL REFERENCES cartao_acesso(num_cartao),
  placa CHAR(7) NOT NULL REFERENCES veiculo(placa),
  guarita_entrada INTEGER NOT NULL REFERENCES guarita(cod_guarita),
  guarita_saida INTEGER REFERENCES guarita(cod_guarita),
  CONSTRAINT saida_depois_entrada CHECK (data_hora_saida > data_hora_entrada),
  CONSTRAINT saida_completa CHECK ((data_hora_saida IS NULL) = (guarita_saida IS NULL))
);
"

CREATE TABLE


### 2.8 Pagamento (avulso) e mensalidade (mensalista)

- **ESTADIA (1) – GERA – (1) PAGAMENTO:** relacionamento **1:1**. A chave estrangeira `cod_estadia` é `UNIQUE`, então uma estadia tem no máximo um pagamento. É `NOT NULL` porque todo pagamento é de uma estadia (participação total).
- **CLIENTE (1) – PAGA – (N) MENSALIDADE:** um cliente só pode ter uma mensalidade por mês (`UNIQUE (cod_cliente, mes_referencia)`).
- Valores sempre positivos (`CHECK`).

In [15]:
%%bash
posql -c "
CREATE TABLE pagamento (
  cod_pagamento SERIAL PRIMARY KEY,
  valor NUMERIC(8,2) NOT NULL CHECK (valor > 0),
  data_hora_pagamento TIMESTAMP NOT NULL,
  forma_pagamento forma_pagamento NOT NULL,
  cod_estadia INTEGER NOT NULL UNIQUE REFERENCES estadia(cod_estadia)
);

CREATE TABLE mensalidade (
  cod_mensalidade SERIAL PRIMARY KEY,
  mes_referencia DATE NOT NULL,
  valor NUMERIC(8,2) NOT NULL CHECK (valor > 0),
  data_vencimento DATE NOT NULL,
  data_pagamento DATE,
  cod_cliente INTEGER NOT NULL REFERENCES cliente(cod_cliente),
  UNIQUE (cod_cliente, mes_referencia)
);
"

CREATE TABLE
CREATE TABLE


### 2.9 Atributo derivado: status da mensalidade

O status **não é guardado**, porque mudaria sozinho com o passar dos dias. Ele é calculado a partir das datas:

- `PAGO`: tem data de pagamento
- `EM ATRASO`: não foi paga e o vencimento já passou
- `PENDENTE`: não foi paga e ainda está no prazo

Para não repetir esse cálculo em toda consulta, criamos uma **view** (uma consulta salva que pode ser usada como se fosse uma tabela).

In [16]:
%%bash
posql -c "
CREATE VIEW vw_mensalidade_status AS
  SELECT m.*,
         CASE
           WHEN m.data_pagamento IS NOT NULL THEN 'PAGO'
           WHEN m.data_vencimento < CURRENT_DATE THEN 'EM ATRASO'
           ELSE 'PENDENTE'
         END AS status
  FROM mensalidade m;
"

CREATE VIEW


Conferindo as tabelas criadas:

In [17]:
!posql -c "\d"

                       List of relations
 Schema |              Name               |   Type   |  Owner   
--------+---------------------------------+----------+----------
 public | cartao_acesso                   | table    | postgres
 public | categoria                       | table    | postgres
 public | categoria_cod_categoria_seq     | sequence | postgres
 public | cliente                         | table    | postgres
 public | cliente_cod_cliente_seq         | sequence | postgres
 public | estadia                         | table    | postgres
 public | estadia_cod_estadia_seq         | sequence | postgres
 public | fabricante                      | table    | postgres
 public | fabricante_cod_fabricante_seq   | sequence | postgres
 public | guarita                         | table    | postgres
 public | guarita_cod_guarita_seq         | sequence | postgres
 public | mensalidade                     | table    | postgres
 public | mensalidade_cod_mensalidade_seq | sequence | postgr

In [18]:
!posql -c "\d estadia"

                                                Table "public.estadia"
      Column       |            Type             | Collation | Nullable |                   Default                    
-------------------+-----------------------------+-----------+----------+----------------------------------------------
 cod_estadia       | integer                     |           | not null | nextval('estadia_cod_estadia_seq'::regclass)
 data_hora_entrada | timestamp without time zone |           | not null | 
 data_hora_saida   | timestamp without time zone |           |          | 
 num_cartao        | integer                     |           | not null | 
 placa             | character(7)                |           | not null | 
 guarita_entrada   | integer                     |           | not null | 
 guarita_saida     | integer                     |           |          | 
Indexes:
    "estadia_pkey" PRIMARY KEY, btree (cod_estadia)
Check constraints:
    "saida_completa" CHECK ((data_hora_s

## 3. População do banco

Os dados representam um mês de funcionamento do estacionamento (outubro de 2026) e alguns veículos que estão no pátio agora. Inserimos sempre informando as colunas (`INSERT INTO tabela (colunas) VALUES ...`), assim os campos `SERIAL` são gerados automaticamente.

In [19]:
%%bash
posql -c "
INSERT INTO fabricante (nome_fabricante) VALUES
  ('Fiat'), ('Volkswagen'), ('Toyota'), ('Mercedes-Benz'),
  ('Honda'), ('Chevrolet'), ('Hyundai');

INSERT INTO modelo (nome_modelo, cod_fabricante) VALUES
  ('Argo', 1), ('Mobi', 1), ('Gol', 2), ('Polo', 2),
  ('Corolla', 3), ('Hilux', 3), ('Sprinter', 4), ('CG 160', 5),
  ('Civic', 5), ('Onix', 6), ('S10', 6), ('HB20', 7);

INSERT INTO categoria (descricao, isenta) VALUES
  ('Carro de passeio', FALSE),
  ('Motocicleta', FALSE),
  ('Caminhonete', FALSE),
  ('Ambulância', TRUE),
  ('Veículo médico', TRUE);

INSERT INTO guarita (localizacao) VALUES
  ('Entrada principal - Av. Central'),
  ('Portaria da emergência'),
  ('Acesso dos funcionários - Rua Lateral');
"

INSERT 0 7
INSERT 0 12
INSERT 0 5
INSERT 0 3


In [20]:
!posql -c "SELECT * FROM modelo"

 cod_modelo | nome_modelo | cod_fabricante 
------------+-------------+----------------
          1 | Argo        |              1
          2 | Mobi        |              1
          3 | Gol         |              2
          4 | Polo        |              2
          5 | Corolla     |              3
          6 | Hilux       |              3
          7 | Sprinter    |              4
          8 | CG 160      |              5
          9 | Civic       |              5
         10 | Onix        |              6
         11 | S10         |              6
         12 | HB20        |              7
(12 rows)



**Clientes (mensalistas).** O telefone é inserido como `ARRAY[...]` (multivalorado) e o endereço como uma tupla `ROW(...)` (composto).

In [21]:
%%bash
posql -c "
INSERT INTO cliente (nome, cpf, telefone, endereco) VALUES
  ('Ana Souza', '12345678901', ARRAY['84999990001', '8432110001'],
   ROW('Rua das Flores', 120, 'Tirol', 'Natal', 'RN', '59020000')),
  ('Carlos Lima', '98765432100', ARRAY['84988880002'],
   ROW('Av. Salgado Filho', 3000, 'Lagoa Nova', 'Natal', 'RN', '59075000')),
  ('Juliana Ramos', '45678912300', ARRAY['84977770003', '84933330003'],
   ROW('Rua Mossoró', 45, 'Petrópolis', 'Natal', 'RN', '59012000')),
  ('Roberto Alves', '32165498700', ARRAY['84966660004'],
   ROW('Rua São José', 800, 'Centro', 'Parnamirim', 'RN', '59140000'));
"

INSERT 0 4


In [22]:
!posql -c "SELECT * FROM cliente"

 cod_cliente |     nome      |     cpf     |         telefone          |                         endereco                          
-------------+---------------+-------------+---------------------------+-----------------------------------------------------------
           1 | Ana Souza     | 12345678901 | {84999990001,8432110001}  | ("Rua das Flores",120,Tirol,Natal,RN,59020000)
           2 | Carlos Lima   | 98765432100 | {84988880002}             | ("Av. Salgado Filho",3000,"Lagoa Nova",Natal,RN,59075000)
           3 | Juliana Ramos | 45678912300 | {84977770003,84933330003} | ("Rua Mossoró",45,Petrópolis,Natal,RN,59012000)
           4 | Roberto Alves | 32165498700 | {84966660004}             | ("Rua São José",800,Centro,Parnamirim,RN,59140000)
(4 rows)



**Veículos.** Os cinco primeiros são de mensalistas (a Ana tem dois carros). Os demais são avulsos (`cod_cliente` nulo), incluindo duas ambulâncias e um veículo médico.

In [23]:
%%bash
posql -c "
INSERT INTO veiculo (placa, cor, ano, cod_modelo, cod_categoria, cod_cliente) VALUES
  ('QWE1A23', 'Prata', 2022, 1, 1, 1),
  ('ANA2B34', 'Branco', 2018, 2, 1, 1),
  ('MTO3D67', 'Vermelha', 2023, 8, 2, 2),
  ('JUL4C56', 'Cinza', 2021, 5, 1, 3),
  ('ROB5D78', 'Preto', 2020, 11, 3, 4),
  ('RNB2C45', 'Branco', 2019, 3, 1, NULL),
  ('TYT4E89', 'Preto', 2020, 6, 3, NULL),
  ('HBX6F12', 'Azul', 2017, 12, 1, NULL),
  ('ONX7G34', 'Branco', 2024, 10, 1, NULL),
  ('PLO8H56', 'Vermelho', 2022, 4, 1, NULL),
  ('CVC9J78', 'Prata', 2016, 9, 1, NULL),
  ('AMB0A01', 'Branco', 2021, 7, 4, NULL),
  ('AMB0A02', 'Branco', 2023, 7, 4, NULL),
  ('MED1K23', 'Branco', 2022, 6, 5, NULL);
"

INSERT 0 14


**Cartões.** O Carlos perdeu o cartão 1002 e recebeu o 1003 (o histórico fica guardado). O cartão do Roberto está bloqueado. Os cartões da série 2000 são avulsos e circulam entre os visitantes.

In [24]:
%%bash
posql -c "
INSERT INTO cartao_acesso (num_cartao, tipo, status, data_emissao, data_validade, cod_cliente) VALUES
  (1001, 'MENSALISTA', 'ATIVO',     '2026-03-01', '2027-03-01', 1),
  (1002, 'MENSALISTA', 'PERDIDO',   '2026-04-01', '2027-04-01', 2),
  (1003, 'MENSALISTA', 'ATIVO',     '2026-08-15', '2027-08-15', 2),
  (1004, 'MENSALISTA', 'ATIVO',     '2026-05-10', '2027-05-10', 3),
  (1005, 'MENSALISTA', 'BLOQUEADO', '2026-02-01', '2027-02-01', 4),
  (2001, 'AVULSO', 'ATIVO', '2026-01-05', NULL, NULL),
  (2002, 'AVULSO', 'ATIVO', '2026-01-05', NULL, NULL),
  (2003, 'AVULSO', 'ATIVO', '2026-01-05', NULL, NULL),
  (2004, 'AVULSO', 'ATIVO', '2026-01-05', NULL, NULL),
  (2005, 'AVULSO', 'ATIVO', '2026-01-05', NULL, NULL),
  (2006, 'AVULSO', 'ATIVO', '2026-01-05', NULL, NULL);
"

INSERT 0 11


In [25]:
!posql -c "SELECT * FROM cartao_acesso"

 num_cartao |    tipo    |  status   | data_emissao | data_validade | cod_cliente 
------------+------------+-----------+--------------+---------------+-------------
       1001 | MENSALISTA | ATIVO     | 2026-03-01   | 2027-03-01    |           1
       1002 | MENSALISTA | PERDIDO   | 2026-04-01   | 2027-04-01    |           2
       1003 | MENSALISTA | ATIVO     | 2026-08-15   | 2027-08-15    |           2
       1004 | MENSALISTA | ATIVO     | 2026-05-10   | 2027-05-10    |           3
       1005 | MENSALISTA | BLOQUEADO | 2026-02-01   | 2027-02-01    |           4
       2001 | AVULSO     | ATIVO     | 2026-01-05   |               |            
       2002 | AVULSO     | ATIVO     | 2026-01-05   |               |            
       2003 | AVULSO     | ATIVO     | 2026-01-05   |               |            
       2004 | AVULSO     | ATIVO     | 2026-01-05   |               |            
       2005 | AVULSO     | ATIVO     | 2026-01-05   |               |            
       2006 | 

**Estadias.** As 13 primeiras já terminaram (outubro de 2026). As 3 últimas estão **no pátio agora**: não têm saída, e a entrada é calculada a partir do horário atual (`NOW() - INTERVAL '...'`).

In [26]:
%%bash
posql -c "
INSERT INTO estadia (data_hora_entrada, data_hora_saida, num_cartao, placa, guarita_entrada, guarita_saida) VALUES
  ('2026-10-01 08:00:00', '2026-10-01 10:30:00', 2001, 'RNB2C45', 1, 1),
  ('2026-10-01 07:10:00', '2026-10-01 19:05:00', 1001, 'QWE1A23', 3, 3),
  ('2026-10-02 13:20:00', '2026-10-02 14:05:00', 2002, 'HBX6F12', 1, 2),
  ('2026-10-03 02:15:00', '2026-10-03 02:55:00', 2003, 'AMB0A01', 2, 2),
  ('2026-10-05 09:00:00', '2026-10-05 17:40:00', 1004, 'JUL4C56', 1, 1),
  ('2026-10-06 10:00:00', '2026-10-06 16:20:00', 2001, 'ONX7G34', 1, 1),
  ('2026-10-08 15:30:00', '2026-10-08 16:10:00', 2004, 'MED1K23', 2, 1),
  ('2026-10-10 07:50:00', '2026-10-10 18:00:00', 1003, 'MTO3D67', 3, 3),
  ('2026-10-12 11:00:00', '2026-10-12 13:30:00', 2002, 'PLO8H56', 1, 1),
  ('2026-10-15 20:00:00', '2026-10-16 08:30:00', 2005, 'CVC9J78', 2, 1),
  ('2026-10-18 08:15:00', '2026-10-18 12:00:00', 1001, 'ANA2B34', 1, 1),
  ('2026-10-20 09:40:00', '2026-10-20 10:20:00', 2003, 'RNB2C45', 1, 2),
  ('2026-10-25 06:30:00', '2026-10-25 07:10:00', 2006, 'AMB0A02', 2, 2),
  (NOW() - INTERVAL '3 hours',  NULL, 2001, 'TYT4E89', 1, NULL),
  (NOW() - INTERVAL '2 hours',  NULL, 1004, 'JUL4C56', 3, NULL),
  (NOW() - INTERVAL '50 minutes', NULL, 2002, 'HBX6F12', 1, NULL);
"

INSERT 0 16


**Pagamentos dos avulsos.** Regra de cobrança: **R$ 5,00 por hora iniciada**. Estadias de mensalistas (2, 5, 8, 11) e de veículos isentos (4, 7, 13) não geram pagamento.

In [27]:
%%bash
posql -c "
INSERT INTO pagamento (valor, data_hora_pagamento, forma_pagamento, cod_estadia) VALUES
  (15.00, '2026-10-01 10:27:00', 'PIX', 1),
  (5.00,  '2026-10-02 14:01:00', 'DINHEIRO', 3),
  (35.00, '2026-10-06 16:15:00', 'CARTAO', 6),
  (15.00, '2026-10-12 13:25:00', 'PIX', 9),
  (65.00, '2026-10-16 08:22:00', 'CARTAO', 10),
  (5.00,  '2026-10-20 10:16:00', 'PIX', 12);
"

INSERT 0 6


**Mensalidades.** Vencimento todo dia 10. A Juliana não pagou outubro, e o Roberto não paga desde setembro (por isso o cartão dele foi bloqueado).

In [28]:
%%bash
posql -c "
INSERT INTO mensalidade (mes_referencia, valor, data_vencimento, data_pagamento, cod_cliente) VALUES
  ('2026-09-01', 250.00, '2026-09-10', '2026-09-08', 1),
  ('2026-10-01', 250.00, '2026-10-10', '2026-10-09', 1),
  ('2026-11-01', 250.00, '2026-11-10', NULL,         1),
  ('2026-09-01', 120.00, '2026-09-10', '2026-09-05', 2),
  ('2026-10-01', 120.00, '2026-10-10', '2026-10-12', 2),
  ('2026-11-01', 120.00, '2026-11-10', NULL,         2),
  ('2026-09-01', 250.00, '2026-09-10', '2026-09-10', 3),
  ('2026-10-01', 250.00, '2026-10-10', NULL,         3),
  ('2026-11-01', 250.00, '2026-11-10', NULL,         3),
  ('2026-09-01', 300.00, '2026-09-10', NULL,         4),
  ('2026-10-01', 300.00, '2026-10-10', NULL,         4);
"

INSERT 0 11


Quantidade de linhas em cada tabela:

In [29]:
%%bash
posql -c "
SELECT 'fabricante' AS tabela, COUNT(*) FROM fabricante UNION ALL
SELECT 'modelo', COUNT(*) FROM modelo UNION ALL
SELECT 'categoria', COUNT(*) FROM categoria UNION ALL
SELECT 'cliente', COUNT(*) FROM cliente UNION ALL
SELECT 'veiculo', COUNT(*) FROM veiculo UNION ALL
SELECT 'cartao_acesso', COUNT(*) FROM cartao_acesso UNION ALL
SELECT 'guarita', COUNT(*) FROM guarita UNION ALL
SELECT 'estadia', COUNT(*) FROM estadia UNION ALL
SELECT 'pagamento', COUNT(*) FROM pagamento UNION ALL
SELECT 'mensalidade', COUNT(*) FROM mensalidade;
"

    tabela     | count 
---------------+-------
 fabricante    |     7
 modelo        |    12
 categoria     |     5
 cliente       |     4
 veiculo       |    14
 cartao_acesso |    11
 guarita       |     3
 estadia       |    16
 pagamento     |     6
 mensalidade   |    11
(10 rows)



## 4. Testes das restrições

Cada comando abaixo **deve dar erro**. É o banco impedindo dados que não fazem sentido no minimundo.

Observação: mesmo quando o `INSERT` falha, o `SERIAL` já gastou o próximo número. Por isso, se novas linhas forem inseridas depois, os códigos vão "pular" alguns valores.

In [30]:
%%bash
# vai dar erro: cartão AVULSO não pode ter cliente (check cartao_tipo_cliente)
posql -c "
INSERT INTO cartao_acesso (num_cartao, tipo, data_emissao, cod_cliente)
VALUES (2099, 'AVULSO', '2026-11-01', 1);
"

ERROR:  new row for relation "cartao_acesso" violates check constraint "cartao_tipo_cliente"
DETAIL:  Failing row contains (2099, AVULSO, ATIVO, 2026-11-01, null, 1).


CalledProcessError: Command 'b'# vai dar erro: cart\xc3\xa3o AVULSO n\xc3\xa3o pode ter cliente (check cartao_tipo_cliente)\nposql -c "\nINSERT INTO cartao_acesso (num_cartao, tipo, data_emissao, cod_cliente)\nVALUES (2099, \'AVULSO\', \'2026-11-01\', 1);\n"\n'' returned non-zero exit status 1.

In [43]:
%%bash
# vai dar erro: o veículo XYZ9Z99 não está cadastrado (chave estrangeira)
posql -c "
INSERT INTO estadia (data_hora_entrada, num_cartao, placa, guarita_entrada)
VALUES ('2026-11-01 10:00:00', 2005, 'XYZ9Z99', 1);
"

ERROR:  insert or update on table "estadia" violates foreign key constraint "estadia_placa_fkey"
DETAIL:  Key (placa)=(XYZ9Z99) is not present in table "veiculo".


CalledProcessError: Command 'b'# vai dar erro: o ve\xc3\xadculo XYZ9Z99 n\xc3\xa3o est\xc3\xa1 cadastrado (chave estrangeira)\nposql -c "\nINSERT INTO estadia (data_hora_entrada, num_cartao, placa, guarita_entrada)\nVALUES (\'2026-11-01 10:00:00\', 2005, \'XYZ9Z99\', 1);\n"\n'' returned non-zero exit status 1.

In [44]:
%%bash
# vai dar erro: a estadia 1 já tem pagamento: relação 1:1 (unique)
posql -c "
INSERT INTO pagamento (valor, data_hora_pagamento, forma_pagamento, cod_estadia)
VALUES (10.00, '2026-10-01 10:40:00', 'PIX', 1);
"

ERROR:  duplicate key value violates unique constraint "pagamento_cod_estadia_key"
DETAIL:  Key (cod_estadia)=(1) already exists.


CalledProcessError: Command 'b'# vai dar erro: a estadia 1 j\xc3\xa1 tem pagamento: rela\xc3\xa7\xc3\xa3o 1:1 (unique)\nposql -c "\nINSERT INTO pagamento (valor, data_hora_pagamento, forma_pagamento, cod_estadia)\nVALUES (10.00, \'2026-10-01 10:40:00\', \'PIX\', 1);\n"\n'' returned non-zero exit status 1.

In [45]:
%%bash
# vai dar erro: a saída não pode ser antes da entrada (check saida_depois_entrada)
posql -c "
UPDATE estadia SET data_hora_saida = '2026-10-01 07:00:00' WHERE cod_estadia = 1;
"

ERROR:  new row for relation "estadia" violates check constraint "saida_depois_entrada"
DETAIL:  Failing row contains (1, 2026-10-01 08:00:00, 2026-10-01 07:00:00, 2001, RNB2C45, 1, 1).


CalledProcessError: Command 'b'# vai dar erro: a sa\xc3\xadda n\xc3\xa3o pode ser antes da entrada (check saida_depois_entrada)\nposql -c "\nUPDATE estadia SET data_hora_saida = \'2026-10-01 07:00:00\' WHERE cod_estadia = 1;\n"\n'' returned non-zero exit status 1.

In [46]:
%%bash
# vai dar erro: placa fora do padrão (check)
posql -c "
INSERT INTO veiculo (placa, cor, ano, cod_modelo, cod_categoria)
VALUES ('abc-123', 'Azul', 2020, 1, 1);
"

ERROR:  new row for relation "veiculo" violates check constraint "veiculo_placa_check"
DETAIL:  Failing row contains (abc-123, Azul, 2020, 1, 1, null).


CalledProcessError: Command 'b'# vai dar erro: placa fora do padr\xc3\xa3o (check)\nposql -c "\nINSERT INTO veiculo (placa, cor, ano, cod_modelo, cod_categoria)\nVALUES (\'abc-123\', \'Azul\', 2020, 1, 1);\n"\n'' returned non-zero exit status 1.

In [47]:
%%bash
# vai dar erro: EXTRAVIADO não é um valor do enum status_cartao
posql -c "
INSERT INTO cartao_acesso (num_cartao, tipo, status, data_emissao)
VALUES (2100, 'AVULSO', 'EXTRAVIADO', '2026-11-01');
"

ERROR:  invalid input value for enum status_cartao: "EXTRAVIADO"
LINE 3: VALUES (2100, 'AVULSO', 'EXTRAVIADO', '2026-11-01');
                                ^


CalledProcessError: Command 'b'# vai dar erro: EXTRAVIADO n\xc3\xa3o \xc3\xa9 um valor do enum status_cartao\nposql -c "\nINSERT INTO cartao_acesso (num_cartao, tipo, status, data_emissao)\nVALUES (2100, \'AVULSO\', \'EXTRAVIADO\', \'2026-11-01\');\n"\n'' returned non-zero exit status 1.

## 5. Consultas (exemplos de utilização)

Cada consulta traz a pergunta que ela responde e, quando possível, a expressão equivalente em álgebra relacional.

### 5.1 Atributo composto e multivalorado

Clientes de Natal, com bairro (parte do endereço composto) e quantidade de telefones (tamanho do array).

$\pi_{\text{nome, bairro, telefones}}(\sigma_{\text{cidade='Natal'}}(\text{cliente}))$

In [31]:
%%bash
posql -c "
SELECT nome,
       (endereco).bairro AS bairro,
       telefone[1] AS telefone_principal,
       ARRAY_LENGTH(telefone, 1) AS qtd_telefones
FROM cliente
WHERE (endereco).cidade = 'Natal';
"

     nome      |   bairro   | telefone_principal | qtd_telefones 
---------------+------------+--------------------+---------------
 Ana Souza     | Tirol      | 84999990001        |             2
 Carlos Lima   | Lagoa Nova | 84988880002        |             1
 Juliana Ramos | Petrópolis | 84977770003        |             2
(3 rows)



### 5.2 Quais veículos estão no pátio agora?

Estadias sem saída, juntando veículo, modelo, categoria, cartão e guarita de entrada.

$\sigma_{\text{data\_hora\_saida IS NULL}}(\text{estadia} \bowtie \text{veiculo} \bowtie \text{modelo} \bowtie \text{categoria} \bowtie \text{cartao\_acesso} \bowtie \text{guarita})$

In [32]:
%%bash
posql -c "
SELECT e.placa, m.nome_modelo, c.descricao AS categoria, ca.tipo AS cartao,
       g.localizacao AS entrou_por,
       DATE_TRUNC('minute', NOW()::TIMESTAMP - e.data_hora_entrada) AS tempo_no_patio
FROM estadia e
  INNER JOIN veiculo v ON e.placa = v.placa
  INNER JOIN modelo m ON v.cod_modelo = m.cod_modelo
  INNER JOIN categoria c ON v.cod_categoria = c.cod_categoria
  INNER JOIN cartao_acesso ca ON e.num_cartao = ca.num_cartao
  INNER JOIN guarita g ON e.guarita_entrada = g.cod_guarita
WHERE e.data_hora_saida IS NULL
ORDER BY e.data_hora_entrada;
"

  placa  | nome_modelo |    categoria     |   cartao   |              entrou_por               | tempo_no_patio 
---------+-------------+------------------+------------+---------------------------------------+----------------
 TYT4E89 | Hilux       | Caminhonete      | AVULSO     | Entrada principal - Av. Central       | 03:01:00
 JUL4C56 | Corolla     | Carro de passeio | MENSALISTA | Acesso dos funcionários - Rua Lateral | 02:01:00
 HBX6F12 | HB20        | Carro de passeio | AVULSO     | Entrada principal - Av. Central       | 00:51:00
(3 rows)



### 5.3 Tempo de permanência (atributo derivado) e valor cobrado

O tempo de permanência é calculado (saída menos entrada). O `LEFT JOIN` com pagamento mantém também as estadias que não geraram cobrança (mensalista ou isento), que aparecem com valor vazio.

In [33]:
%%bash
posql -c "
SELECT e.cod_estadia, e.placa, ca.tipo AS cartao, c.isenta,
       e.data_hora_saida - e.data_hora_entrada AS tempo_permanencia,
       p.valor AS valor_pago
FROM estadia e
  INNER JOIN cartao_acesso ca ON e.num_cartao = ca.num_cartao
  INNER JOIN veiculo v ON e.placa = v.placa
  INNER JOIN categoria c ON v.cod_categoria = c.cod_categoria
  LEFT JOIN pagamento p ON p.cod_estadia = e.cod_estadia
WHERE e.data_hora_saida IS NOT NULL
ORDER BY e.cod_estadia;
"

 cod_estadia |  placa  |   cartao   | isenta | tempo_permanencia | valor_pago 
-------------+---------+------------+--------+-------------------+------------
           1 | RNB2C45 | AVULSO     | f      | 02:30:00          |      15.00
           2 | QWE1A23 | MENSALISTA | f      | 11:55:00          |           
           3 | HBX6F12 | AVULSO     | f      | 00:45:00          |       5.00
           4 | AMB0A01 | AVULSO     | t      | 00:40:00          |           
           5 | JUL4C56 | MENSALISTA | f      | 08:40:00          |           
           6 | ONX7G34 | AVULSO     | f      | 06:20:00          |      35.00
           7 | MED1K23 | AVULSO     | t      | 00:40:00          |           
           8 | MTO3D67 | MENSALISTA | f      | 10:10:00          |           
           9 | PLO8H56 | AVULSO     | f      | 02:30:00          |      15.00
          10 | CVC9J78 | AVULSO     | f      | 12:30:00          |      65.00
          11 | ANA2B34 | MENSALISTA | f      | 03:45:00       

### 5.4 Situação das mensalidades (atributo derivado)

Usando a view criada na seção 2.9.

In [34]:
%%bash
posql -c "
SELECT c.nome, TO_CHAR(s.mes_referencia, 'MM/YYYY') AS mes, s.valor,
       s.data_vencimento, s.data_pagamento, s.status
FROM vw_mensalidade_status s
  INNER JOIN cliente c ON s.cod_cliente = c.cod_cliente
ORDER BY c.nome, s.mes_referencia;
"

     nome      |   mes   | valor  | data_vencimento | data_pagamento |  status   
---------------+---------+--------+-----------------+----------------+-----------
 Ana Souza     | 09/2026 | 250.00 | 2026-09-10      | 2026-09-08     | PAGO
 Ana Souza     | 10/2026 | 250.00 | 2026-10-10      | 2026-10-09     | PAGO
 Ana Souza     | 11/2026 | 250.00 | 2026-11-10      |                | PENDENTE
 Carlos Lima   | 09/2026 | 120.00 | 2026-09-10      | 2026-09-05     | PAGO
 Carlos Lima   | 10/2026 | 120.00 | 2026-10-10      | 2026-10-12     | PAGO
 Carlos Lima   | 11/2026 | 120.00 | 2026-11-10      |                | PENDENTE
 Juliana Ramos | 09/2026 | 250.00 | 2026-09-10      | 2026-09-10     | PAGO
 Juliana Ramos | 10/2026 | 250.00 | 2026-10-10      |                | EM ATRASO
 Juliana Ramos | 11/2026 | 250.00 | 2026-11-10      |                | PENDENTE
 Roberto Alves | 09/2026 | 300.00 | 2026-09-10      |                | EM ATRASO
 Roberto Alves | 10/2026 | 300.00 | 2026-10-10      | 

### 5.5 Quais clientes estão inadimplentes?

$\pi_{\text{nome, telefone}}(\sigma_{\text{status='EM ATRASO'}}(\text{vw\_mensalidade\_status}) \bowtie \text{cliente})$

O `DISTINCT` remove repetições (o Roberto tem dois meses em atraso), como a projeção da álgebra relacional.

In [35]:
%%bash
posql -c "
SELECT DISTINCT c.nome, c.telefone
FROM vw_mensalidade_status s
  INNER JOIN cliente c ON s.cod_cliente = c.cod_cliente
WHERE s.status = 'EM ATRASO';
"

     nome      |         telefone          
---------------+---------------------------
 Juliana Ramos | {84977770003,84933330003}
 Roberto Alves | {84966660004}
(2 rows)



### 5.6 Histórico de um veículo, com guarita de entrada e de saída (renomeação)

A tabela `guarita` aparece **duas vezes** na mesma consulta (uma para a entrada, outra para a saída). Para diferenciar, renomeamos cada uma com `AS`.

$\rho_{\text{ge}}(\text{guarita})$ e $\rho_{\text{gs}}(\text{guarita})$

In [36]:
%%bash
posql -c "
SELECT e.cod_estadia, e.data_hora_entrada, ge.localizacao AS guarita_entrada,
       e.data_hora_saida, gs.localizacao AS guarita_saida
FROM estadia e
  INNER JOIN guarita AS ge ON e.guarita_entrada = ge.cod_guarita
  LEFT JOIN guarita AS gs ON e.guarita_saida = gs.cod_guarita
WHERE e.placa = 'RNB2C45'
ORDER BY e.data_hora_entrada;
"

 cod_estadia |  data_hora_entrada  |         guarita_entrada         |   data_hora_saida   |          guarita_saida          
-------------+---------------------+---------------------------------+---------------------+---------------------------------
           1 | 2026-10-01 08:00:00 | Entrada principal - Av. Central | 2026-10-01 10:30:00 | Entrada principal - Av. Central
          12 | 2026-10-20 09:40:00 | Entrada principal - Av. Central | 2026-10-20 10:20:00 | Portaria da emergência
(2 rows)



### 5.7 Estadias de veículos isentos (gratuidade)

$\sigma_{\text{isenta = true}}(\text{estadia} \bowtie \text{veiculo} \bowtie \text{categoria})$

In [37]:
%%bash
posql -c "
SELECT e.cod_estadia, v.placa, c.descricao AS categoria, e.data_hora_entrada, e.data_hora_saida
FROM estadia e
  INNER JOIN veiculo v ON e.placa = v.placa
  INNER JOIN categoria c ON v.cod_categoria = c.cod_categoria
WHERE c.isenta;
"

 cod_estadia |  placa  |   categoria    |  data_hora_entrada  |   data_hora_saida   
-------------+---------+----------------+---------------------+---------------------
           4 | AMB0A01 | Ambulância     | 2026-10-03 02:15:00 | 2026-10-03 02:55:00
           7 | MED1K23 | Veículo médico | 2026-10-08 15:30:00 | 2026-10-08 16:10:00
          13 | AMB0A02 | Ambulância     | 2026-10-25 06:30:00 | 2026-10-25 07:10:00
(3 rows)



### 5.8 União: veículos que passaram pela portaria da emergência

Veículos que **entraram** pela guarita 2 $\cup$ veículos que **saíram** por ela.

In [38]:
%%bash
posql -c "
SELECT placa FROM estadia WHERE guarita_entrada = 2
UNION
SELECT placa FROM estadia WHERE guarita_saida = 2;
"

  placa  
---------
 RNB2C45
 AMB0A02
 AMB0A01
 HBX6F12
 CVC9J78
 MED1K23
(6 rows)



### 5.9 Interseção: mensalistas inadimplentes que estão no pátio agora

Clientes com veículo no pátio $\cap$ clientes com mensalidade em atraso. Usamos uma tabela temporária, como em aula, para deixar a consulta mais legível.

In [39]:
%%bash
posql -c "
CREATE TEMP TABLE inadimplentes_no_patio AS
  SELECT v.cod_cliente FROM estadia e
    INNER JOIN veiculo v ON e.placa = v.placa
  WHERE e.data_hora_saida IS NULL
  INTERSECT
  SELECT cod_cliente FROM vw_mensalidade_status WHERE status = 'EM ATRASO';

SELECT c.nome, c.telefone
FROM inadimplentes_no_patio i
  INNER JOIN cliente c ON i.cod_cliente = c.cod_cliente;
"

SELECT 1
     nome      |         telefone          
---------------+---------------------------
 Juliana Ramos | {84977770003,84933330003}
(1 row)



### 5.10 Subtração: veículos cadastrados que nunca estacionaram

$\pi_{\text{placa}}(\text{veiculo}) - \pi_{\text{placa}}(\text{estadia})$

In [40]:
%%bash
posql -c "
SELECT placa FROM veiculo
EXCEPT
SELECT placa FROM estadia;
"

  placa  
---------
 ROB5D78
(1 row)



### 5.11 Faturamento por mês

Soma dos pagamentos avulsos e das mensalidades pagas, agrupados por mês (`GROUP BY` com as funções de agregação `COUNT` e `SUM`).

In [41]:
%%bash
posql -c "
SELECT TO_CHAR(data_hora_pagamento, 'YYYY-MM') AS mes, 'AVULSO' AS origem,
       COUNT(*) AS quantidade, SUM(valor) AS total
FROM pagamento
GROUP BY TO_CHAR(data_hora_pagamento, 'YYYY-MM')
UNION ALL
SELECT TO_CHAR(data_pagamento, 'YYYY-MM'), 'MENSALIDADE', COUNT(*), SUM(valor)
FROM mensalidade
WHERE data_pagamento IS NOT NULL
GROUP BY TO_CHAR(data_pagamento, 'YYYY-MM')
ORDER BY mes, origem;
"

   mes   |   origem    | quantidade | total  
---------+-------------+------------+--------
 2026-09 | MENSALIDADE |          3 | 620.00
 2026-10 | AVULSO      |          6 | 140.00
 2026-10 | MENSALIDADE |          2 | 370.00
(3 rows)



### 5.12 Cartões que não estão ativos

In [42]:
%%bash
posql -c "
SELECT ca.num_cartao, ca.tipo, ca.status, c.nome AS titular
FROM cartao_acesso ca
  LEFT JOIN cliente c ON ca.cod_cliente = c.cod_cliente
WHERE ca.status <> 'ATIVO';
"

 num_cartao |    tipo    |  status   |    titular    
------------+------------+-----------+---------------
       1002 | MENSALISTA | PERDIDO   | Carlos Lima
       1005 | MENSALISTA | BLOQUEADO | Roberto Alves
(2 rows)

